<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_06_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 06 Lab: Retrieval-Augmented Generation (RAG)
**AI Search Over Custom Data · COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

Run the cells **from top to bottom, in order**. Parts 1–5 run locally and need no API key. Parts 6 and 8 call the NRP: your key must be saved in Colab Secrets (🔑) as `NRP_API_KEY` with notebook access turned on (see Module 00).

| Part | Model |
|---|---|
| 6 – Response generation | `qwen3-small` (thinking off) |
| 8 – Hallucination analysis | `gpt-oss` (thinking low) |

## Setup

In [ ]:
!pip install -q -U chromadb openai

## Part 1 – Dataset Preparation
The fifth document is a **made-up fact** (the Aurora-7 drone does not exist). No model can know it from training, so it shows clearly whether an answer came from retrieval.

In [ ]:
documents = [
    "Artificial intelligence is used in healthcare for diagnosis and treatment.",
    "Machine learning is a subset of AI focused on learning from data.",
    "Neural networks are inspired by the human brain.",
    "RAG systems combine retrieval with generation.",
    "The Aurora-7 drone flies for 42 minutes per charge."
]

print(f"Number of documents: {len(documents)}")

## Part 2 – Chunking Strategy

In [ ]:
def chunk_text(text, chunk_size=50):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc, chunk_size=10)
    all_chunks.extend(chunks)

print(f"Original Documents: {len(documents)}")
print(f"Processed Chunks: {len(all_chunks)}")

for i, chunk in enumerate(all_chunks):
    print(f"{i}: {chunk}")

## Part 3 – Embeddings and Vector Database

In [ ]:
import chromadb
chroma_client = chromadb.Client()   # named chroma_client so it is not replaced in Part 6

try:
    chroma_client.delete_collection(name="local_rag_collection")
except:
    pass

collection = chroma_client.create_collection(name="local_rag_collection")

collection.add(
    documents=all_chunks,
    ids=[f"id_{i}" for i in range(len(all_chunks))]
)
print("Part 3 Complete: Chunks vectorized locally and stored in RAM.")
print("Stored chunks:", collection.count())

## Part 3B – Inspect Stored Documents and Embeddings

In [ ]:
all_retrieved_items = collection.get(
    ids=[f"id_{i}" for i in range(collection.count())],
    include=['documents', 'embeddings']
)

for i in range(len(all_retrieved_items['ids'])):
    doc = all_retrieved_items['documents'][i]
    embedding = all_retrieved_items['embeddings'][i]
    print(f"Document: '{doc}'")
    print(f"Embedding (first 5 values): {embedding[:5]}...\n")

## Part 4 – Similarity Search

In [ ]:
query_text = "How is AI used in healthcare?"

results = collection.query(
    query_texts=[query_text],
    n_results=2
)

print("--- Search Results ---")
for i, doc in enumerate(results["documents"][0]):
    print(f"Match {i+1}: {doc}")

## Part 5 – RAG Prompt Construction

In [ ]:
retrieved_docs = results["documents"][0]
context = "\n".join(retrieved_docs)

augmented_prompt = f"""
You are a research assistant. Using ONLY the facts provided in the CONTEXT below,
answer the question. If you don't know the answer, just say that you don't know.
Include every relevant detail found in the context.

CONTEXT:
{context}

QUESTION: {query_text}
"""

print("--- Augmented Prompt ---")
print(augmented_prompt)

## Part 6 – Response Generation with the NRP

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("NRP_API_KEY").strip(),
                base_url="https://ellm.nrp-nautilus.io/v1")

response = client.chat.completions.create(
    model="qwen3-small",
    messages=[
        {"role": "system",
         "content": "You are a helpful assistant. Use ONLY the provided context to answer."},
        {"role": "user", "content": augmented_prompt}
    ],
    temperature=0,
    extra_body={"reasoning_effort": "none"}   # thinking off: answer only from the context
)

print("--- NRP AI RESPONSE ---")
print(response.choices[0].message.content)

## Part 7 – Retrieval Tuning
Shows the retrieved chunks for each Top-k, then generates an answer from each.

In [ ]:
# Example tuning experiment
for k in [1, 2, 4]:
    results = collection.query(query_texts=[query_text], n_results=k)
    print(f"\nTop-k = {k}")
    for doc in results["documents"][0]:
        print("-", doc)

In [ ]:
# Generate an answer for each Top-k so you can compare answer quality.
import time
for k in [1, 2, 4]:
    results = collection.query(query_texts=[query_text], n_results=k)
    context = "\n".join(results["documents"][0])
    prompt = (f"Using ONLY the facts in the CONTEXT, answer the question. If you don't know, say so.\n\n"
              f"CONTEXT:\n{context}\n\nQUESTION: {query_text}")
    response = client.chat.completions.create(
        model="qwen3-small",
        messages=[{"role": "system", "content": "Use ONLY the provided context to answer."},
                  {"role": "user", "content": prompt}],
        temperature=0, extra_body={"reasoning_effort": "none"})
    print(f"\n=== Top-k = {k} ===\n{response.choices[0].message.content}")
    time.sleep(2)   # share the NRP service fairly

## Part 8 – Hallucination Analysis
Three questions, each asked **without** RAG and **with** RAG. `build_rag_prompt()` retrieves fresh context for each question.
1. **General knowledge:** both answers should be similar.
2. **Private fact (Aurora-7):** only the RAG answer can be grounded; watch whether the non-RAG answer guesses.
3. **Out of context:** the RAG answer should say it does not know, because the documents do not cover it.

In [ ]:
def build_rag_prompt(question, k=2):
    """Retrieve context for THIS question and wrap it in the RAG prompt."""
    results = collection.query(query_texts=[question], n_results=k)
    context = "\n".join(results["documents"][0])
    return f"""
You are a research assistant. Using ONLY the facts provided in the CONTEXT below,
answer the question. If you don't know the answer, just say that you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""

def ask(messages):
    response = client.chat.completions.create(
        model="gpt-oss", messages=messages, temperature=0,
        extra_body={"reasoning_effort": "low"})   # gpt-oss cannot turn thinking off
    return response.choices[0].message.content

test_queries = [
    "What specifically are neural networks inspired by?",   # general knowledge
    "How long can the Aurora-7 drone fly on one charge?",    # only in our documents
    "What is the capital of Australia?",                     # not in our documents
]
for test_query in test_queries:
    print(f"\n===== {test_query} =====")
    print("--- TEST 1: WITHOUT RAG ---")
    print(ask([{"role": "user", "content": test_query}]))
    print("--- TEST 2: WITH RAG ---")
    print(ask([{"role": "system", "content": "Use ONLY the provided context."},
               {"role": "user", "content": build_rag_prompt(test_query)}]))

## Optional Extension – Compare with NRP Embeddings
It comes after Part 8 because it needs the NRP connection created in Part 6. Embeds the same chunks with the NRP's `qwen3-embedding` model and compares retrieval with ChromaDB's local embeddings. If this cell returns an error, skip it.

In [ ]:
# Optional: embed the same chunks with the NRP's qwen3-embedding model.
def nrp_embed(texts):
    response = client.embeddings.create(model="qwen3-embedding", input=texts)
    return [item.embedding for item in response.data]

nrp_collection = chroma_client.create_collection(name="nrp_rag_collection")
nrp_collection.add(documents=all_chunks, embeddings=nrp_embed(all_chunks),
                   ids=[f"id_{i}" for i in range(len(all_chunks))])

question = "How is AI used in healthcare?"
local = collection.query(query_texts=[question], n_results=2)
remote = nrp_collection.query(query_embeddings=nrp_embed([question]), n_results=2)
print("Local embeddings:", local["documents"][0])
print("NRP embeddings:  ", remote["documents"][0])

## Part 9 – Evaluation
Use your outputs from Parts 4–8 to fill in the experiment log below (edit the text in the table, then run the cell).

In [ ]:
import pandas as pd
log = pd.DataFrame([
    {"Part": 4, "Question / Query": "How is AI used in healthcare?", "Top-k": "2",
     "Retrieved Context Present?": "", "Model Response Summary": "", "Observation": ""},
    {"Part": 7, "Question / Query": "How is AI used in healthcare?", "Top-k": "1/2/4",
     "Retrieved Context Present?": "", "Model Response Summary": "", "Observation": ""},
    {"Part": 8, "Question / Query": "What specifically are neural networks inspired by?", "Top-k": "RAG vs. non-RAG",
     "Retrieved Context Present?": "", "Model Response Summary": "", "Observation": ""},
    {"Part": 8, "Question / Query": "How long can the Aurora-7 drone fly on one charge?", "Top-k": "RAG vs. non-RAG",
     "Retrieved Context Present?": "", "Model Response Summary": "", "Observation": ""},
    {"Part": 8, "Question / Query": "What is the capital of Australia? (out of context)", "Top-k": "2",
     "Retrieved Context Present?": "", "Model Response Summary": "", "Observation": ""},
])
log.to_csv("module06_experiment_log.csv", index=False)
log

## Before you submit
- [ ] Parts 1–8 ran from top to bottom (the embeddings extension is optional).
- [ ] The experiment log is filled in.
- [ ] This notebook does not contain your API key.
- [ ] Zip this notebook with your Technical Reflection Report.